# Notebook 08: Advanced Training Techniques

## Overview

- **Duration**: ~2 hours
- **Prerequisites**: Notebooks 01-07
- **Learning Objectives**:
  1. Implement mixed precision training (FP16/BF16)
  2. Implement gradient accumulation
  3. Use Flash Attention for faster training
  4. Understand distributed training basics

In [1]:
import sys
sys.path.insert(0, '../..')

import math
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm.auto import tqdm

print(f"PyTorch Cuda version: {torch.version.cuda}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Check for bfloat16 support
bf16_supported = device.type == "cuda" and torch.cuda.is_bf16_supported()
print(f"BFloat16 supported: {bf16_supported}")

PyTorch Cuda version: 12.8
Device: cuda
BFloat16 supported: True


In [2]:
try:
    from shared.model import GPT, GPTConfig
    print("Using GPT from shared.model")

except ImportError as e:
    print("Could not import GPT from shared.model, defining it here.")
    #define GPT
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import math


    class GPTConfig:
        def __init__(self, vocab_size, block_size, n_layer=4, n_head=4, n_embd=128, dropout=0.0, bias=False):
            self.vocab_size = vocab_size
            self.block_size = block_size
            self.n_layer = n_layer
            self.n_head = n_head
            self.n_embd = n_embd
            self.dropout = dropout
            self.bias = bias


    class CausalSelfAttention(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.attn = nn.MultiheadAttention(
                embed_dim=config.n_embd,
                num_heads=config.n_head,
                batch_first=True
            )
            self.register_buffer(
                "mask",
                torch.tril(torch.ones(config.block_size, config.block_size))
            )

        def forward(self, x):
            B, T, C = x.size()
            mask = self.mask[:T, :T]
            x, _ = self.attn(x, x, x, attn_mask=(mask == 0))
            return x


    class Block(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.ln1 = nn.LayerNorm(config.n_embd)
            self.attn = CausalSelfAttention(config)
            self.ln2 = nn.LayerNorm(config.n_embd)
            self.mlp = nn.Sequential(
                nn.Linear(config.n_embd, 4 * config.n_embd),
                nn.GELU(),
                nn.Linear(4 * config.n_embd, config.n_embd),
            )

        def forward(self, x):
            x = x + self.attn(self.ln1(x))
            x = x + self.mlp(self.ln2(x))
            return x


    class GPT(nn.Module):
        def __init__(self, config):
            super().__init__()
            self.config = config

            self.token_emb = nn.Embedding(config.vocab_size, config.n_embd)
            self.pos_emb = nn.Embedding(config.block_size, config.n_embd)

            self.blocks = nn.Sequential(
                *[Block(config) for _ in range(config.n_layer)]
            )

            self.ln_f = nn.LayerNorm(config.n_embd)
            self.head = nn.Linear(config.n_embd, config.vocab_size)

        def forward(self, idx, targets=None):
            B, T = idx.shape
            device = idx.device

            pos = torch.arange(0, T, device=device)

            tok = self.token_emb(idx)
            pos = self.pos_emb(pos)

            x = tok + pos
            x = self.blocks(x)
            x = self.ln_f(x)

            logits = self.head(x)

            loss = None
            if targets is not None:
                B, T, C = logits.shape
                logits = logits.view(B * T, C)
                targets = targets.view(B * T)
                loss = F.cross_entropy(logits, targets)

            return logits, loss

Could not import GPT from shared.model, defining it here.


## Exercise 8.1: Mixed Precision Training (30 min)

Mixed precision uses FP16 or BF16 for most operations:
- **Faster**: More ops per second on modern GPUs
- **Less memory**: Smaller activations = larger batches
- **Same accuracy**: Accumulate in FP32

In [3]:
####solution 8.1
def train_step_mixed_precision(
    model: nn.Module,
    x: torch.Tensor,
    y: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    grad_clip: float = 1.0,
    dtype: torch.dtype = torch.bfloat16,
):
    """
    Single training step with mixed precision.

    Args:
        model: The model
        x, y: Input and target tensors
        optimizer: The optimizer
        scaler: GradScaler for FP16 (not needed for BF16)
        grad_clip: Gradient clipping value
        dtype: torch.float16 or torch.bfloat16

    Returns:
        loss value
    """
    # TODO: Implement mixed precision training step
    # 1. Use torch.autocast context for forward pass
    #    with torch.autocast(device_type="cuda", dtype=dtype):
    #        logits, loss = model(x, y)
    with torch.autocast(device_type="cuda", dtype=dtype):
        logits, loss = model(x, y)

    # 2. For FP16: scale loss and backward
    #    scaler.scale(loss).backward()
    #    For BF16: just loss.backward()
    if dtype == torch.float16:
        scaler.scale(loss).backward()
    if dtype == torch.bfloat16:
        loss.backward()

    # 3. Unscale and clip gradients
    #    scaler.unscale_(optimizer)  # For FP16
    #    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    if dtype == torch.float16:
        scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

    # 4. Optimizer step
    #    scaler.step(optimizer)  # For FP16
    #    scaler.update()         # For FP16
    #    optimizer.step()        # For BF16
    if dtype == torch.float16:
        scaler.step(optimizer)
        scaler.update()
    if dtype == torch.bfloat16:
        optimizer.step()
    optimizer.zero_grad()

    return loss.item()

In [4]:
from tqdm.auto import tqdm
####test cell
# Benchmark mixed precision vs FP32
config = GPTConfig(vocab_size=10000, block_size=256, n_layer=6, n_head=6, n_embd=384)
model = GPT(config).to(device)

x = torch.randint(0, config.vocab_size, (32, 256), device=device)
y = torch.randint(0, config.vocab_size, (32, 256), device=device)

# Warmup
for _ in range(3):
    _, loss = model(x, y)
    loss.backward()
    model.zero_grad()

# Benchmark FP32
#torch.cuda.synchronize()
start = time.time()
pbar1 = tqdm(range(20), desc="FP32 benchmark")
for _ in pbar1:
    _, loss = model(x, y)
    loss.backward()
    model.zero_grad()
#torch.cuda.synchronize()
fp32_time = time.time() - start

# Benchmark BF16
#torch.cuda.synchronize()
start = time.time()
pbar2 = tqdm(range(20), desc="BF16 benchmark")
for _ in pbar2:
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        _, loss = model(x, y)
    loss.backward()
    model.zero_grad()
#torch.cuda.synchronize()
bf16_time = time.time() - start

print(f"FP32 time: {fp32_time:.3f}s")
print(f"BF16 time: {bf16_time:.3f}s")
print(f"Speedup: {fp32_time/bf16_time:.2f}x")

FP32 benchmark:   0%|          | 0/20 [00:00<?, ?it/s]

BF16 benchmark:   0%|          | 0/20 [00:00<?, ?it/s]

FP32 time: 4.808s
BF16 time: 4.853s
Speedup: 0.99x


## Exercise 8.2: Gradient Accumulation (30 min)

Accumulate gradients over multiple mini-batches to simulate larger batch sizes.

In [5]:
####solution 8.2
def train_with_gradient_accumulation(
    model: nn.Module,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    accumulation_steps: int = 4,
    max_steps: int = 100,
    grad_clip: float = 1.0,
):
    """
    Train with gradient accumulation.

    Effective batch size = batch_size * accumulation_steps
    """
    model.train()
    data_iter = iter(dataloader)
    losses = []

    for step in tqdm(range(max_steps)):
        accumulated_loss = 0.0

        # TODO: Implement gradient accumulation
        # 1. Loop for accumulation_steps
        for _ in range(accumulation_steps):
            # 2. Get batch (cycle dataloader if needed)
            try:
                batch = next(data_iter)
            except StopIteration:
                data_iter = iter(dataloader)
                batch = next(data_iter)

            # 3. Forward pass with autocast
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16):
                _, loss = model(batch['input_ids'], batch['labels'])

            # 4. Scale loss by accumulation_steps
            loss = loss / accumulation_steps
            accumulated_loss += loss.item()

            # 5. Backward (accumulate gradients)
            loss.backward()


        # 6. After accumulation: clip grads, optimizer step, zero grads
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        optimizer.zero_grad()
        losses.append(accumulated_loss)

    return losses


## Exercise 8.3: Flash Attention (Optional)

PyTorch 2.0+ includes Flash Attention via `torch.nn.functional.scaled_dot_product_attention`.

In [6]:
# Check Flash Attention availability
import torch.nn.functional as F

# PyTorch 2.0+ uses Flash Attention automatically when possible
print(f"PyTorch version: {torch.__version__}")

# Test scaled_dot_product_attention
q = torch.randn(2, 8, 256, 64, device=device)
k = torch.randn(2, 8, 256, 64, device=device)
v = torch.randn(2, 8, 256, 64, device=device)

# This will use Flash Attention if available
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    out = F.scaled_dot_product_attention(q, k, v, is_causal=True)

print(f"Output shape: {out.shape}")

PyTorch version: 2.10.0+cu128
Output shape: torch.Size([2, 8, 256, 64])


## Complete Training Loop with All Optimizations

In [7]:
def train_optimized(
    model,
    train_loader,
    val_loader,
    max_steps=500,
    warmup_steps=50,
    max_lr=3e-4,
    min_lr=3e-5,
    accumulation_steps=1,
    use_amp=True,
):
    """Optimized training loop with mixed precision and gradient accumulation."""
    from shared.training import get_lr, configure_optimizer

    optimizer = configure_optimizer(model, 0.1, max_lr, (0.9, 0.95))
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and not bf16_supported))
    dtype = torch.bfloat16 if bf16_supported else torch.float16

    model.train()
    data_iter = iter(train_loader)
    history = {"train_loss": [], "val_loss": []}

    pbar = tqdm(range(max_steps))
    for step in pbar:
        # LR schedule
        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr

        # Accumulation loop
        optimizer.zero_grad(set_to_none=True)
        acc_loss = 0.0

        for _ in range(accumulation_steps):
            try:
                x, y = next(data_iter)
            except StopIteration:
                data_iter = iter(train_loader)
                x, y = next(data_iter)

            x, y = x.to(device), y.to(device)

            with torch.autocast(device_type="cuda", dtype=dtype, enabled=use_amp):
                _, loss = model(x, y)
                loss = loss / accumulation_steps

            scaler.scale(loss).backward()
            acc_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if step % 10 == 0:
            history["train_loss"].append(acc_loss)
            pbar.set_postfix({"loss": f"{acc_loss:.4f}", "lr": f"{lr:.2e}"})

    return history

## Summary

### What You Learned

1. **Mixed Precision**: BF16/FP16 for faster training
2. **Gradient Accumulation**: Simulate larger batches
3. **Flash Attention**: Memory-efficient attention (PyTorch 2.0+)

### GPU Memory Tips

| Technique | Memory Savings | Speed Impact |
|-----------|---------------|---------------|
| BF16/FP16 | ~50% | +20-50% faster |
| Gradient Accumulation | Proportional | Slight overhead |
| Flash Attention | ~50% for attention | +20-40% faster |

### Next: Text Generation

In Notebook 09, we'll generate text with our trained model!